# Reporting and Adapter Bridges

SCPN Phase Orchestrator provides:

- **CoherencePlot**: Diagnostic visualisation from audit JSONL logs
- **Adapter bridges**: Type-safe converters between SPO and external systems

This notebook demonstrates:

1. Generating a simulation with audit log data
2. CoherencePlot: R timeline, regime timeline, action audit
3. FusionCoreBridge: fusion equilibrium → phase-orchestrator
4. PlasmaControlBridge: scpn-control telemetry → UPDEState
5. SCPNControlBridge: Knm/omega import and state export

In [ ]:
import numpy as np

from scpn_phase_orchestrator.coupling.knm import CouplingBuilder
from scpn_phase_orchestrator.upde.engine import UPDEEngine
from scpn_phase_orchestrator.upde.metrics import LayerState, UPDEState
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter

## 1. Generate audit log data for plotting

In [ ]:
N = 8
DT = 0.01
STEPS = 200

rng = np.random.default_rng(7)
omegas = rng.uniform(0.8, 1.2, N)
coupling = CouplingBuilder().build(N, base_strength=0.5, decay_alpha=0.2)
alpha = coupling.alpha.copy()
phases = rng.uniform(0, 2 * np.pi, N)
engine = UPDEEngine(N, DT)

log_data = []
regime = "NOMINAL"

for step in range(STEPS):
    if step == 80:
        phases = rng.uniform(0, 2 * np.pi, N)

    phases = engine.step(phases, omegas, coupling.knm, 0.0, 0.0, alpha)
    R, psi = compute_order_parameter(phases)

    if R < 0.3:
        regime = "CRITICAL"
    elif R < 0.6:
        regime = "DEGRADED"
    else:
        regime = "NOMINAL"

    actions = []
    if R < 0.4 and step % 10 == 0:
        actions = [{"knob": "K", "scope": "global", "value": 1.2}]

    log_data.append(
        {
            "step": step,
            "layers": [{"R": R, "psi": psi}],
            "regime": regime,
            "actions": actions,
        }
    )

last = log_data[-1]
last_r = last["layers"][0]["R"]
print(f"Generated {len(log_data)} step records")
print(f"Final R: {last_r:.4f}, regime: {last['regime']}")

## 2. CoherencePlot: Diagnostic Visualisation

In [ ]:
import tempfile
from pathlib import Path

from scpn_phase_orchestrator.reporting.plots import CoherencePlot

plot = CoherencePlot(log_data)
tmpdir = Path(tempfile.mkdtemp())

r_path = plot.plot_r_timeline(tmpdir / "r_timeline.png")
print(f"R timeline saved: {r_path}")

regime_path = plot.plot_regime_timeline(tmpdir / "regime_timeline.png")
print(f"Regime timeline saved: {regime_path}")

action_path = plot.plot_action_audit(tmpdir / "action_audit.png")
print(f"Action audit saved: {action_path}")

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib.image import imread

    paths = [r_path, regime_path, action_path]
    titles = ["R Timeline", "Regime Timeline", "Action Audit"]
    fig, axes = plt.subplots(3, 1, figsize=(10, 10))
    for ax, path, title in zip(axes, paths, titles, strict=True):
        ax.imshow(imread(str(path)))
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available, check saved PNGs manually")

## 3. FusionCoreBridge: Equilibrium → Phases

In [ ]:
from scpn_phase_orchestrator.adapters import FusionCoreBridge

bridge = FusionCoreBridge(n_layers=6)

snapshot = {
    "q_profile": 2.5,
    "q_min": 1.0,
    "q_max": 5.0,
    "beta_n": 1.8,
    "tau_e": 2.1,
    "sawtooth_count": 3,
    "elm_count": 7,
    "mhd_amplitude": 0.4,
}

phases = bridge.observables_to_phases(snapshot)
obs_names = ["q_profile", "beta_n", "tau_e", "sawtooth", "elm", "mhd"]
print(f"Fusion phases ({len(phases)} oscillators):")
for name, ph in zip(obs_names, phases, strict=True):
    print(f"  {name}: {ph:.4f} rad ({np.degrees(ph):.1f}deg)")

omegas_fusion = np.ones(6)
feedback = bridge.phases_to_feedback(phases, omegas_fusion)
r_g = feedback["R_global"]
m_p = feedback["mean_phase"]
print(f"\nFeedback: R={r_g:.4f}, mean_phase={m_p:.4f}")

violations = bridge.check_stability({"q_min": 0.8, "beta_n": 3.2})
print(f"\nStability violations: {len(violations)}")
for v in violations:
    print(f"  {v['variable']}: {v['message']}")

## 4. PlasmaControlBridge: Telemetry → UPDEState

In [ ]:
from scpn_phase_orchestrator.adapters import PlasmaControlBridge

plasma = PlasmaControlBridge(n_layers=4)

omega_plasma = plasma.import_plasma_omega()
print("Plasma layer frequencies (rad/s):")
names = ["micro_turbulence", "zonal_flow", "mhd_tearing", "sawtooth_elm"]
for name, w in zip(names, omega_plasma, strict=True):
    print(f"  {name}: {w:.1f}")

layer_knm = np.array(
    [
        [0.0, 0.5, 0.1, 0.0],
        [0.5, 0.0, 0.3, 0.1],
        [0.1, 0.3, 0.0, 0.4],
        [0.0, 0.1, 0.4, 0.0],
    ]
)
coupling_state = plasma.import_knm_spec(
    {
        "matrix": layer_knm,
        "n_osc_per_layer": 2,
    }
)
print(f"\nExpanded Knm: {coupling_state.knm.shape}")

tick_result = {
    "phases": np.array([0.1, 0.2, 1.5, 1.6, 3.0, 3.1, 5.0, 5.2]),
    "regime": "NOMINAL",
    "layer_sizes": [2, 2, 2, 2],
}
state = plasma.import_snapshot(tick_result)
print("\nImported UPDEState:")
print(f"  Regime: {state.regime_id}")
print(f"  Layers: {len(state.layers)}")
for i, ls in enumerate(state.layers):
    print(f"    L{i}: R={ls.R:.4f}, psi={ls.psi:.4f}")

violations = plasma.check_physics_invariants(
    {
        "q_min": 0.7,
        "greenwald": 1.5,
    }
)
print(f"\nPhysics violations: {len(violations)}")
for v in violations:
    print(f"  {v['message']}")

## 5. SCPNControlBridge: State Import/Export

In [ ]:
from scpn_phase_orchestrator.adapters import SCPNControlBridge

scpn = SCPNControlBridge(scpn_config={"mode": "feedback"})

ext_knm = np.array([[0.0, 0.3], [0.3, 0.0]])
cs = scpn.import_knm(ext_knm)
print(f"Imported coupling: template={cs.active_template}, shape={cs.knm.shape}")

ext_omega = np.array([1.0, 1.5])
omega = scpn.import_omega(ext_omega)
print(f"Imported omega: {omega}")

upde_state = UPDEState(
    layers=[LayerState(R=0.85, psi=1.2), LayerState(R=0.72, psi=2.8)],
    cross_layer_alignment=np.eye(2),
    stability_proxy=0.78,
    regime_id="NOMINAL",
)
export = scpn.export_state(upde_state)
print("\nExported telemetry:")
print(f"  regime: {export['regime']}")
print(f"  stability: {export['stability']}")
print(f"  layers: {len(export['layers'])}")
for i, lay in enumerate(export["layers"]):
    print(f"    L{i}: R={lay['R']}, psi={lay['psi']:.4f}")

## Summary

- **CoherencePlot** reads JSONL audit data and produces R timeline, regime timeline, action audit, amplitude, and PAC heatmap PNGs
- **FusionCoreBridge** converts 6 fusion equilibrium observables to [0, 2π) phases and checks Troyon/Kruskal-Shafranov stability
- **PlasmaControlBridge** provides Kronecker coupling expansion, plasma-timescale omegas, and UPDEState import from scpn-control
- **SCPNControlBridge** wraps external Knm/omega imports and exports UPDEState as telemetry dicts

Additional adapters: `QuantumControlBridge`, `SNNControllerBridge`, `OTelExporter`.